# 031 — Chiller disconnect forensics

Reproduces the `chiller_service` housekeeping/communication logic 1:1 (same
`Chiller` class, same channels, same 15 s interval, **no plots**) and captures
forensic evidence at the exact moment a chiller stops responding.

Background (KB `memory/devices/chiller.md`, quirk OPEN): repeated
"disconnects", *"Das Gerät erkennt den Befehl nicht"* = **WinError 22**,
per-device Reconnect helps only briefly, lab-master restart fixes durably.
Windows energy-saving options were already adjusted (2026-07-14) and the
disconnects persist. Competing theories this notebook discriminates:

1. **USB driver wedge** — the adapter handle dies at driver level (WinError 22).
2. **Protocol desync** — one timed-out read leaves a late reply in the buffer;
   every following reply is off by one command.
3. **Blocking write** — production sets no `write_timeout`; a wedged adapter
   hangs the thread silently.

## Prerequisites

- **Stop `chiller_service` first** (System card on the dashboard, or stop
  lab-master). One COM port = one owning process — this notebook cannot open
  ports the service still holds.
- Run under the **esibd** conda env kernel.

## Deliberate deviations from production (diagnostic-only)

1. `write_timeout = 2 s` (production blocks forever) — turns a silent write
   hang into a logged `SerialTimeoutException`.
2. A housekeeping cycle continues past a failed read (production aborts the
   whole cycle) — shows whether ALL commands fail or only some.
3. On failure: state snapshot (full exception chain incl. `winerror`, buffer
   levels, stale-byte hexdump, Windows COM enumeration check) + a staged
   recovery probe ladder.
4. **No telemetry sink** — this run writes nothing into telemetry.db.

In [1]:
import sys
import re
import time
import logging
from pathlib import Path
from datetime import datetime

# Add project source to Python path
project_root = Path.cwd().parent.parent
src_path = project_root / "src"
sys.path.insert(0, str(src_path))

print(f"Project root: {project_root}")

try:
    from devices.chiller.chiller import Chiller, ChillerCommands
    import serial
    from serial.tools import list_ports
    print("✅ Imports OK")
except ImportError as e:
    print(f"❌ Import failed: {e}")

Project root: C:\Users\ESIBDlab\Desktop\LAB_Code\esibd_bs
✅ Imports OK


In [2]:
# Ports come from the canonical [com_ports] in lab_services/lab_config.toml.
# COM numbers shift across reboots — never hardcode them here.
LAB_CONFIG = project_root.parent / "lab_services" / "lab_config.toml"

def load_chiller_ports(path):
    text = Path(path).read_text(encoding="utf-8")
    try:
        import tomllib
        ports = tomllib.loads(text)["com_ports"]
    except ModuleNotFoundError:  # Python < 3.11 fallback
        ports = {m.group(1): int(m.group(2)) for m in
                 re.finditer(r"^(Chiller_[A-C])\s*=\s*(\d+)", text, re.M)}
    return {k: f"COM{v}" for k, v in ports.items() if k.startswith("Chiller_")}

CHILLER_PORTS = load_chiller_ports(LAB_CONFIG)
print(f"Chiller ports from lab_config.toml: {CHILLER_PORTS}")

# Uncomment to debug a subset / override manually:
# CHILLER_PORTS = {"Chiller_B": "COM19"}

# --- Tunables ---------------------------------------------------------------
HK_INTERVAL_S = 15.0             # same as chiller_service (lab_config hk_interval)
SERIAL_TIMEOUT_S = 1.0           # same as production (Chiller default)
WRITE_TIMEOUT_S = 2.0            # DEVIATION: production blocks forever on write
PROBE_ON_FAILURE = True          # run the recovery probe ladder automatically
PROBE_AFTER_N_FAILED_CYCLES = 3  # let a few raw failures get logged first
OK_HEARTBEAT_EVERY = 20          # INFO "still fine" marker every N good cycles

Chiller ports from lab_config.toml: {'Chiller_A': 'COM23', 'Chiller_B': 'COM19', 'Chiller_C': 'COM20'}


In [3]:
logs_dir = project_root / "debugging" / "logs"
logs_dir.mkdir(parents=True, exist_ok=True)
run_ts = datetime.now().strftime("%Y%m%d_%H%M%S")
log_path = logs_dir / f"031_chiller_disconnect_{run_ts}.log"

diag = logging.getLogger("chiller_disconnect_debug")
diag.setLevel(logging.DEBUG)
diag.handlers.clear()

fh = logging.FileHandler(log_path, encoding="utf-8")
fh.setLevel(logging.DEBUG)   # file gets EVERY serial transaction (raw bytes + timing)
ch = logging.StreamHandler()
ch.setLevel(logging.INFO)    # console: lifecycle + failures; set WARNING to mute hk lines
fmt = logging.Formatter("%(asctime)s.%(msecs)03d %(levelname)-7s %(message)s",
                        datefmt="%Y-%m-%d %H:%M:%S")
fh.setFormatter(fmt)
ch.setFormatter(fmt)
diag.addHandler(fh)
diag.addHandler(ch)

print(f"Diagnostic log: {log_path}")


def exc_chain(e):
    """Full exception chain with errno/winerror — winerror=22 is the USB-driver signature."""
    parts, seen, cur = [], set(), e
    while cur is not None and id(cur) not in seen:
        seen.add(id(cur))
        s = f"{type(cur).__name__}: {cur}"
        extras = [f"{a}={getattr(cur, a)}" for a in ("errno", "winerror")
                  if getattr(cur, a, None) is not None]
        if extras:
            s += f" [{', '.join(extras)}]"
        parts.append(s)
        cur = cur.__cause__ or cur.__context__
    return "  <-  ".join(parts)


# Reverse map command string -> constant name, for readable log labels
COMMAND_NAMES = {v: k for k, v in vars(ChillerCommands).items()
                 if isinstance(v, str) and not k.startswith("_")}

Diagnostic log: C:\Users\ESIBDlab\Desktop\LAB_Code\esibd_bs\debugging\logs\031_chiller_disconnect_20260714_090044.log


In [4]:
class DiagnosticChiller(Chiller):
    """Chiller with production-identical protocol logic plus forensic logging.

    Deviations from production (each deliberate, diagnostic-only): see the
    intro cell. Everything else — commands, parsing, channels, locking —
    is the stock Chiller class.
    """

    _HK_READS = (
        ("Cur_Temp", "read_temp", "degC", ".2f"),
        ("Set_Temp", "read_set_temp", "degC", ".2f"),
        ("Run_Stat", "read_running", "", ""),
        ("Dev_Stat", "read_status", "", ""),
        ("Pump_Lvl", "read_pump_level", "", ""),
        ("Col_Stat", "read_cooling", "", ""),
    )

    def __init__(self, *args, diag, **kwargs):
        self.diag = diag
        self._cycle_count = 0
        self._fail_count = 0        # consecutive failed hk cycles
        self._total_failures = 0    # failed hk cycles this run
        self._first_fail_ts = None
        self._probe_done_for_episode = False
        super().__init__(*args, **kwargs)

    # --- transport -----------------------------------------------------

    def _open_transport(self):
        super()._open_transport()
        # DEVIATION 1: production leaves write_timeout at pyserial's default
        # (block forever). A wedged adapter then hangs the thread silently;
        # with a timeout it raises SerialTimeoutException = loggable evidence.
        self.serial_connection.write_timeout = WRITE_TIMEOUT_S
        self.diag.info(
            f"{self.device_id} | port opened: {self.serial_connection.port} "
            f"baud={self.serial_connection.baudrate} "
            f"timeout={self.serial_connection.timeout} "
            f"write_timeout={self.serial_connection.write_timeout}"
        )

    # --- instrumented serial I/O -----------------------------------------

    def _check_stale(self, label):
        """Log residual buffer bytes before a write — desync evidence. Never raises.
        Caller must hold thread_lock."""
        try:
            pre_in = self.serial_connection.in_waiting
            pre_out = self.serial_connection.out_waiting
        except Exception as e:
            self.diag.error(
                f"{self.device_id} | {label}: buffer query FAILED before write "
                f"(ClearCommError = handle dead at DRIVER level): {exc_chain(e)}"
            )
            return
        if pre_in:
            self.diag.warning(
                f"{self.device_id} | {label}: {pre_in} STALE byte(s) in input buffer "
                f"BEFORE write — desync evidence (late reply from an earlier "
                f"timed-out read?)"
            )
        if pre_out:
            self.diag.warning(
                f"{self.device_id} | {label}: out_waiting={pre_out} before write — "
                f"previous write never left the adapter (wedge evidence)"
            )

    def read_dev(self, command):
        label = COMMAND_NAMES.get(command, repr(command))
        with self.thread_lock:
            if not self.serial_connection or not self.serial_connection.is_open:
                raise Exception("Serial connection not open")
            self._check_stale(label)
            t0 = time.perf_counter()
            try:
                self.serial_connection.write(command.encode("ascii"))
            except Exception as e:
                self.diag.error(
                    f"{self.device_id} | {label}: write FAILED after "
                    f"{(time.perf_counter() - t0) * 1e3:.0f} ms: {exc_chain(e)}"
                )
                raise
            t1 = time.perf_counter()
            try:
                raw = self.serial_connection.readline()
            except Exception as e:
                self.diag.error(
                    f"{self.device_id} | {label}: read FAILED after "
                    f"{(time.perf_counter() - t1) * 1e3:.0f} ms: {exc_chain(e)}"
                )
                raise
            t2 = time.perf_counter()
            self.diag.debug(
                f"{self.device_id} | {label}: {command!r} -> {raw!r} "
                f"(write {(t1 - t0) * 1e3:.0f} ms, read {(t2 - t1) * 1e3:.0f} ms)"
            )
            if raw == b"":
                self.diag.warning(
                    f"{self.device_id} | {label}: EMPTY response — read timeout after "
                    f"{self.timeout}s, device did not answer"
                )
            try:
                return raw.decode("ascii").strip()  # strict, same as production
            except UnicodeDecodeError as e:
                self.diag.error(
                    f"{self.device_id} | {label}: non-ASCII garbage in response "
                    f"{raw!r} (hex: {raw.hex(' ')}): {exc_chain(e)}"
                )
                raise

    def set_param(self, param):
        if self.test_mode:
            return super().set_param(param)
        label = f"SET {param.split(' ')[0]}"
        with self.thread_lock:
            if not self.serial_connection or not self.serial_connection.is_open:
                raise Exception("Serial connection not open")
            self._check_stale(label)
            command = f"{param}\r\n"
            t0 = time.perf_counter()
            try:
                self.serial_connection.write(command.encode("ascii"))
                raw = self.serial_connection.readline()
            except Exception as e:
                self.diag.error(
                    f"{self.device_id} | {label}: write/read FAILED: {exc_chain(e)}"
                )
                raise
            self.diag.debug(
                f"{self.device_id} | {label}: {command!r} -> {raw!r} "
                f"({(time.perf_counter() - t0) * 1e3:.0f} ms)"
            )
            response = raw.decode("ascii").strip()
            if response != "OK":
                self.diag.error(
                    f"{self.device_id} | {label}: device did NOT ack (expected 'OK', "
                    f"got {response!r}) — desync or device-side rejection"
                )
                raise Exception(f"Failed to set parameter {param}. Response: {response}")

    # --- housekeeping ------------------------------------------------------

    def hk_monitor(self):
        self._cycle_count += 1
        n = self._cycle_count
        t_start = time.perf_counter()
        errors = []
        results = {}
        # DEVIATION 2: per-read capture; production aborts the cycle on the
        # first exception — here we keep going to see the failure pattern.
        for channel, reader_name, unit, fmt in self._HK_READS:
            try:
                value = getattr(self, reader_name)()
                results[channel] = value
                self.log_sample(channel, value, unit, fmt=fmt)
                if channel == "Run_Stat" and value is not None:
                    self.log_sample("Running", 1 if value == "DEVICE RUNNING" else 0)
            except Exception as e:
                errors.append((channel, e))
                self.diag.error(
                    f"{self.device_id} | cycle #{n} {channel} read FAILED: {exc_chain(e)}"
                )
        # Mapped reads that parsed to None: the raw integer was outside the
        # expected map — classic off-by-one desync symptom (the reply belonged
        # to a different command).
        for channel in ("Run_Stat", "Dev_Stat", "Col_Stat"):
            if channel in results and results[channel] is None:
                self.diag.warning(
                    f"{self.device_id} | cycle #{n} {channel} parsed to None — value "
                    f"outside the expected map, possible desync"
                )
        dur_ms = (time.perf_counter() - t_start) * 1e3
        if errors:
            if self._fail_count == 0:
                self._first_fail_ts = time.time()
                self._probe_done_for_episode = False
            self._fail_count += 1
            self._total_failures += 1
            self.log_event(
                "error",
                f"hk cycle #{n}: {len(errors)}/{len(self._HK_READS)} reads failed "
                f"({dur_ms:.0f} ms)",
            )
            try:
                self.failure_snapshot(errors)
            except Exception as e:
                self.diag.error(f"{self.device_id} | snapshot itself failed: {exc_chain(e)}")
            if (PROBE_ON_FAILURE and not self._probe_done_for_episode
                    and self._fail_count >= PROBE_AFTER_N_FAILED_CYCLES):
                self._probe_done_for_episode = True
                try:
                    self.probe_ladder()
                except Exception as e:
                    self.diag.error(
                        f"{self.device_id} | probe ladder itself failed: {exc_chain(e)}"
                    )
        else:
            if self._fail_count:
                outage = time.time() - self._first_fail_ts
                self.diag.info(
                    f"{self.device_id} | RECOVERED at cycle #{n} after "
                    f"{self._fail_count} failed cycle(s) (~{outage:.0f} s outage)"
                )
            self._fail_count = 0
            self._first_fail_ts = None
            if n % OK_HEARTBEAT_EVERY == 0:
                self.diag.info(
                    f"{self.device_id} | cycle #{n} OK ({dur_ms:.0f} ms, "
                    f"Cur_Temp={results.get('Cur_Temp')})"
                )

    # --- forensics -----------------------------------------------------------

    def failure_snapshot(self, errors=()):
        """Dump everything knowable about the port at the moment of failure."""
        d = self.diag
        d.error("=" * 78)
        d.error(
            f"FAILURE SNAPSHOT {self.device_id} ({self.port}) — consecutive failed "
            f"cycles: {self._fail_count}, total this run: {self._total_failures}"
        )
        if self._first_fail_ts:
            d.error(
                f"  failing since {datetime.fromtimestamp(self._first_fail_ts):%H:%M:%S} "
                f"(~{time.time() - self._first_fail_ts:.0f} s)"
            )
        for channel, e in errors:
            d.error(f"  {channel}: {exc_chain(e)}")
        sc = self.serial_connection
        if sc is None:
            d.error("  serial_connection is None")
        else:
            d.error(
                f"  port object: is_open={sc.is_open} timeout={sc.timeout} "
                f"write_timeout={sc.write_timeout}"
            )
            with self.thread_lock:
                try:
                    iw, ow = sc.in_waiting, sc.out_waiting
                    note = " — pending write stuck in adapter (wedge evidence)" if ow else ""
                    d.error(f"  buffers: in_waiting={iw} out_waiting={ow}{note}")
                    if iw:
                        stale = sc.read(iw)
                        d.error(
                            f"  drained {len(stale)} stale byte(s): {stale!r} | "
                            f"hex: {stale.hex(' ')}"
                        )
                except Exception as e:
                    d.error(
                        f"  buffer query/drain FAILED (ClearCommError -> handle dead at "
                        f"DRIVER level, the WinError-22 signature): {exc_chain(e)}"
                    )
        try:
            present = {p.device: p for p in list_ports.comports()}
            if self.port in present:
                p = present[self.port]
                d.error(f"  {self.port} still enumerated: {p.description} | hwid={p.hwid}")
            else:
                d.error(
                    f"  {self.port} GONE from Windows COM enumeration — USB "
                    f"re-enumeration/driver drop! Present now: {sorted(present)}"
                )
                d.error("  -> check Windows Event Viewer (System log) at this timestamp")
        except Exception as e:
            d.error(f"  COM enumeration failed: {exc_chain(e)}")
        d.error("=" * 78)

    def probe_ladder(self):
        """Staged recovery attempt; WHICH step recovers = WHICH theory holds.

        STEP 1  flush buffers + retry       -> recovery = protocol desync
        STEP 2  close/reopen port + retry   -> recovery = stale handle
                (this replicates the dashboard Reconnect button)
        nothing recovers -> driver wedged beyond in-process repair
                (matches 'only lab-master restart helps')
        """
        d = self.diag
        d.info(f"{self.device_id} | PROBE LADDER start")
        with self.thread_lock:
            try:
                self.serial_connection.reset_input_buffer()
                self.serial_connection.reset_output_buffer()
                d.info(f"{self.device_id} | STEP 1: buffers flushed")
            except Exception as e:
                d.error(f"{self.device_id} | STEP 1: flush FAILED: {exc_chain(e)}")
        try:
            temp = float(self.read_dev(ChillerCommands.READ_TEMP))
            d.info(
                f"{self.device_id} | STEP 1 RECOVERED (Cur_Temp={temp}) -> DESYNC "
                f"theory: buffers were poisoned, the port itself is fine"
            )
            return "step1"
        except Exception as e:
            d.info(f"{self.device_id} | STEP 1 retry still failing: {exc_chain(e)}")
        with self.thread_lock:
            try:
                self._close_transport()
                d.info(f"{self.device_id} | STEP 2: port closed")
            except Exception as e:
                d.warning(
                    f"{self.device_id} | STEP 2: close failed (continuing): {exc_chain(e)}"
                )
            try:
                self._open_transport()
            except Exception as e:
                d.error(
                    f"{self.device_id} | STEP 2: REOPEN FAILED -> handle/driver dead "
                    f"beyond in-process recovery (matches 'only lab-master restart "
                    f"helps'): {exc_chain(e)}"
                )
                self.is_connected = False
                return "dead"
        try:
            temp = float(self.read_dev(ChillerCommands.READ_TEMP))
            d.info(
                f"{self.device_id} | STEP 2 RECOVERED (Cur_Temp={temp}) -> STALE-HANDLE "
                f"theory: close/reopen fixed it (matches 'Reconnect helps briefly')"
            )
            return "step2"
        except Exception as e:
            d.error(
                f"{self.device_id} | STEP 2: port reopened but device still mute -> "
                f"adapter wedged deeper, or device side (power/cable/EMI): {exc_chain(e)}"
            )
            return "mute"

print("✅ DiagnosticChiller defined")

✅ DiagnosticChiller defined


In [5]:
# Baseline: full COM inventory with hardware ids (diff target for later snapshots)
diag.info("USB/COM baseline:")
for p in sorted(list_ports.comports(), key=lambda p: p.device):
    diag.info(f"  {p.device}: {p.description} | hwid={p.hwid}")

chillers = {}
for device_id, port in CHILLER_PORTS.items():
    c = DiagnosticChiller(
        device_id=device_id,
        port=port,
        timeout=SERIAL_TIMEOUT_S,
        hk_interval=HK_INTERVAL_S,
        logger=diag,   # canonical hk lines and diag lines share one file -> one timeline
        diag=diag,
    )
    ok = c.connect()
    print(f"{device_id} on {port}: {'✅ connected' if ok else '❌ CONNECT FAILED (chiller_service still running?)'}")
    chillers[device_id] = c

2026-07-14 09:00:51.351 INFO    USB/COM baseline:
2026-07-14 09:00:51.470 INFO      COM1: Communications Port (COM1) | hwid=ACPI\PNP0501\0
2026-07-14 09:00:51.470 INFO      COM10: Enhanced/PCI - Serial Port(COM10) | hwid=SBMP\*PNP0501\8&39DE33EE&0&007
2026-07-14 09:00:51.471 INFO      COM11: Enhanced/PCI - Serial Port(COM11) | hwid=SBMP\*PNP0501\8&39DE33EE&0&008
2026-07-14 09:00:51.472 INFO      COM12: Enhanced/PCI - Serial Port(COM12) | hwid=SBMP\*PNP0501\8&39DE33EE&0&009
2026-07-14 09:00:51.473 INFO      COM13: Enhanced/PCI - Serial Port(COM13) | hwid=SBMP\*PNP0501\8&39DE33EE&0&010
2026-07-14 09:00:51.473 INFO      COM14: Enhanced/PCI - Serial Port(COM14) | hwid=SBMP\*PNP0501\8&39DE33EE&0&011
2026-07-14 09:00:51.475 INFO      COM15: Enhanced/PCI - Serial Port(COM15) | hwid=SBMP\*PNP0501\8&39DE33EE&0&012
2026-07-14 09:00:51.475 INFO      COM16: Enhanced/PCI - Serial Port(COM16) | hwid=SBMP\*PNP0501\8&39DE33EE&0&013
2026-07-14 09:00:51.476 INFO      COM17: Enhanced/PCI - Serial Port(CO

Chiller_A on COM23: ✅ connected
Chiller_B on COM19: ✅ connected
Chiller_C on COM20: ✅ connected


In [6]:
for c in chillers.values():
    c.start_housekeeping()

2026-07-14 09:00:55.692 INFO    Chiller_A     COM23  housekeeping worker started
2026-07-14 09:00:55.692 INFO    Chiller_A     COM23  housekeeping started (internal mode, interval 15.0s)
2026-07-14 09:00:55.695 INFO    Chiller_B     COM19  housekeeping worker started
2026-07-14 09:00:55.695 INFO    Chiller_B     COM19  housekeeping started (internal mode, interval 15.0s)
2026-07-14 09:00:55.698 INFO    Chiller_C     COM20  housekeeping worker started
2026-07-14 09:00:55.699 INFO    Chiller_C     COM20  housekeeping started (internal mode, interval 15.0s)
2026-07-14 09:00:55.702 INFO    Chiller_A     COM23  Cur_Temp             16.01 degC
2026-07-14 09:00:55.702 INFO    Chiller_B     COM19  Cur_Temp             18.19 degC
2026-07-14 09:00:55.703 INFO    Chiller_C     COM20  Cur_Temp             25.10 degC


Leave this running as long as it takes to catch an episode (hours are fine —
the hk threads are daemons, the kernel stays responsive). Everything goes to
the console (INFO+) and the timestamped log file (DEBUG: every raw serial
transaction with timing). Watch for `FAILURE SNAPSHOT` blocks and the
`PROBE LADDER` verdict.

The cells below are re-runnable at any time while housekeeping runs.

In [14]:
# Re-run anytime: one-line health per chiller
for device_id, c in chillers.items():
    s = c.get_status()
    print(
        f"{device_id}: connected={s['connected']} responding={s['responding']} "
        f"last_sample_age_s={s['last_sample_age_s']} cycles={c._cycle_count} "
        f"failing_now={c._fail_count} failed_total={c._total_failures}"
    )
print(f"\nLog file: {log_path}")

Chiller_A: connected=True responding=True last_sample_age_s=10.9 cycles=61 failing_now=0 failed_total=3
Chiller_B: connected=True responding=True last_sample_age_s=11.0 cycles=61 failing_now=0 failed_total=3
Chiller_C: connected=True responding=True last_sample_age_s=11.1 cycles=61 failing_now=0 failed_total=3

Log file: C:\Users\ESIBDlab\Desktop\LAB_Code\esibd_bs\debugging\logs\031_chiller_disconnect_20260714_090044.log


In [16]:
# On-demand forensics while a chiller is misbehaving (thread-safe, the
# per-transaction lock keeps serial exchanges atomic):

chillers["Chiller_B"].failure_snapshot()
chillers["Chiller_B"].probe_ladder()

2026-07-14 09:16:39.892 ERROR   ==============================================================================
2026-07-14 09:16:39.893 ERROR   FAILURE SNAPSHOT Chiller_B (COM19) — consecutive failed cycles: 1, total this run: 4
2026-07-14 09:16:39.893 ERROR     failing since 09:16:12 (~28 s)
2026-07-14 09:16:39.894 ERROR     port object: is_open=False timeout=1.0 write_timeout=2.0
2026-07-14 09:16:39.894 ERROR     buffer query/drain FAILED (ClearCommError -> handle dead at DRIVER level, the WinError-22 signature): SerialException: ClearCommError failed (OSError(9, 'Das Handle ist ungültig.', None, 6))
2026-07-14 09:16:39.924 ERROR     COM19 still enumerated: USB Serial Device (COM19) | hwid=USB VID:PID=20E3:0001 SER=37FFD50553503431 LOCATION=1-5
2026-07-14 09:16:39.925 ERROR   ==============================================================================
2026-07-14 09:16:39.926 INFO    Chiller_B | PROBE LADDER start
2026-07-14 09:16:39.926 ERROR   Chiller_B | STEP 1: flush FAILED: Port

'step2'

In [ ]:
# OPTIONAL — exercises the WRITE path. The 2026-07-12/13 episodes correlated
# with setpoint writes from /water; this rewrites the CURRENT setpoint (no
# actual change) through the same OUT_SP_00 command, fully instrumented.

# c = chillers["Chiller_B"]
# sp = c.read_set_temp()
# print(f"current setpoint: {sp}")
# c.set_temperature(sp)

In [15]:
# Shutdown: stop housekeeping + release the COM ports (then restart
# chiller_service from the dashboard System card).
for device_id, c in chillers.items():
    c.disconnect()
    print(f"{device_id}: released")

2026-07-14 09:16:21.542 INFO    Chiller_A     COM23  housekeeping worker stopped
2026-07-14 09:16:21.543 INFO    Chiller_A     COM23  housekeeping stopped (internal mode)
2026-07-14 09:16:21.546 INFO    Chiller_A     COM23  disconnected
2026-07-14 09:16:21.547 INFO    Chiller_B     COM19  housekeeping worker stopped
2026-07-14 09:16:21.547 INFO    Chiller_B     COM19  housekeeping stopped (internal mode)
2026-07-14 09:16:21.548 INFO    Chiller_B     COM19  disconnected
2026-07-14 09:16:21.548 INFO    Chiller_C     COM20  housekeeping worker stopped
2026-07-14 09:16:21.550 INFO    Chiller_C     COM20  housekeeping stopped (internal mode)
2026-07-14 09:16:21.552 INFO    Chiller_C     COM20  disconnected


Chiller_A: released
Chiller_B: released
Chiller_C: released


## Reading the evidence

| Log signature | Meaning |
|---|---|
| `winerror=22` in an exception chain, or `buffer query FAILED (ClearCommError ...)` | USB adapter handle died at **driver level** (theory 1) |
| `STALE byte(s) ... BEFORE write`, `parsed to None`, `could not convert string to float` while the port stays open | **protocol desync** (theory 2) — STEP 1 recovery confirms |
| `write FAILED ... SerialTimeoutException`, or `out_waiting` stuck > 0 | **wedged adapter blocking writes** (theory 3) |
| `EMPTY response — read timeout` repeatedly, STEP 2 recovery works | stale handle — matches "Reconnect helps briefly" |
| `GONE from Windows COM enumeration` | USB re-enumerated — correlate Event Viewer (System log) at that timestamp |
| `STEP 2: REOPEN FAILED` | wedged beyond in-process repair — matches "only lab-master restart helps" |
| All three chillers fail within the same minute | shared cause: USB hub, driver, or EMI (compressor kick) — not a single adapter |

## After a caught episode

1. Note the wall-clock time of the first `FAILURE SNAPSHOT`.
2. Windows Event Viewer → System log: USB/serial driver entries at that time.
3. Device Manager → each USB-serial adapter → Power Management →
   uncheck "Allow the computer to turn off this device" — **also on the USB
   Root Hubs / Generic USB Hubs**. This per-device checkbox is separate from
   the power-plan energy options already changed on 2026-07-14.
4. Power plan → advanced → USB settings → USB selective suspend → Disabled
   (if not already done).
5. File findings on the chiller KB page (`memory/devices/chiller.md`, quirk
   OPEN).